# Tarefa 3 — MLP: NumPy + PyTorch + scikit-learn (um só notebook)

**Kernel:** `masters-curriculum` · **Dados:** `../../../data/liver.csv` · **Prazo:** 07/09 23:59

Este arquivo junta **três versões** do mesmo experimento. Depois você edita/encurta para o PDF final.

| Bloco | O quê | Status esperado |
| --- | --- | --- |
| **0** | Dados + padronização + protocolo (compartilhado) | você já fez no NumPy |
| **1** | MLP **NumPy** (do zero) | feito — refine partições/PDF |
| **2** | MLP **PyTorch** (mesmo protocolo) | a preencher (bem mais curto) |
| **3** | MLP **scikit-learn** (`MLPClassifier`) | a preencher (mais curto ainda) |
| **S** | Tabela comparativa + discussão | preencher números |

**Sem Manim** neste notebook (remoção a pedido).

Como usar: rode o Bloco 0+1 (já deve funcionar), depois complete 2 e 3 com as mesmas `SEEDS` / split / alvo Turney.


## A0. NumPy mínimo (rode isto antes de qualquer TODO)

Você só precisa destas ideias agora. O resto vem depois.

| Ideia | Código | Em português |
| --- | --- | --- |
| Tabela | `A.shape` | quantas linhas × colunas |
| Média por coluna | `np.mean(A, axis=0)` | “ao longo das linhas” |
| Desvio por coluna | `np.std(A, axis=0)` | espalhamento de cada exame |
| Misturar features | `A @ w` | combinação linear |
| Par a par | `a * b` | multiplica posição a posição |
| Cortar extremos | `np.clip(x, lo, hi)` | nada abaixo de `lo` nem acima de `hi` |

**Analogia:** um array é uma planilha. `.shape` é o tamanho da planilha. `@` é “mistura colunas com pesos”. `*` é “célula com célula”.


## A. Aquecimento NumPy (10 minutos)

Esqueça MLP por enquanto. Só queremos lembrar: **array = tabela de números**, e `.shape` diz o tamanho.

Convenção neste curso:

- `X` tem shape `(n, d)` → `n` linhas (pessoas), `d` colunas (exames).
- `y` tem shape `(n,)` → um rótulo por pessoa (`0` ou `1`).
- `@` ou `np.dot` = multiplicação de matrizes.
- `*` = multiplicação **elemento a elemento** (mesmo tamanho).

**Analogia (papel):**

| Operação | Pergunta | Uso típico |
| --- | --- | --- |
| `@` | “Como combino **todas** as features com pesos?” | `X @ W`, `A @ w` |
| `*` | “Como ajusto **cada** número com o seu par?” | `a * (1 - a)`, máscaras |

`@` mistura colunas. `*` alinha posição a posição.


In [42]:
import numpy as np
from pathlib import Path

# Exemplo já pronto — só rode e leia o print
A = np.array([[1.0, 2.0],
              [3.0, 4.0],
              [5.0, 6.0]])
print("A.shape =", A.shape)   # (3, 2) → 3 linhas, 2 colunas

w = np.array([0.5, -1.0])
print("w.shape =", w.shape)   # (2,)

# Combinação linear de cada linha de A com w: (3,2) @ (2,) → (3,)
z = A @ w
print("z =", z, "shape", z.shape)

# Somar um bias (número) a cada entrada de z
b = 0.1
z2 = z + b
print("z2 =", z2)


A.shape = (3, 2)
w.shape = (2,)
z = [-1.5 -2.5 -3.5] shape (3,)
z2 = [-1.4 -2.4 -3.4]


### TODO A1 — média por coluna

**Entrada:** `X` com shape `(n, d)`  
**Saída:** `mean` com shape `(d,)` — a média de cada exame  
**Ideia:** `np.mean(X, axis=0)` (o `0` = “ao longo das linhas”).

**Analogia:** `axis=0` pergunta “qual a média **de cada coluna**?”. Sem `axis`, a pergunta vira “qual a média de **todos** os números juntos?” — misture exames e perde o sentido.

Substitua o `...`.


In [43]:
X_demo = np.array([[1.0, 10.0],
                   [3.0, 30.0],
                   [5.0, 50.0]])

# TODO A1
mean = np.mean(X_demo, axis=0)  # deve dar aproximadamente array([3., 30.])

print(mean)
assert np.allclose(mean, [3.0, 30.0])


[ 3. 30.]


### TODO A2 — padronizar uma matriz

Queremos cada coluna com média ≈ 0 e desvio ≈ 1:

$$
X_{\mathrm{std}} = \frac{X - \mathrm{mean}}{\mathrm{std}}
$$

**Analogia (duas etapas):**

| Etapa | Pergunta | Ação |
| --- | --- | --- |
| Centralizar | “Onde está o zero desta coluna?” | subtrai `mean` |
| Escala | “Quão larga é esta coluna?” | divide por `std` |

Sem a escala, um exame em dezenas “grita” mais que um exame em unidades — a MLP “ouve” a escala, não só o sinal clínico.

**Cuidado:** se `std` for 0, some um epsilon (`1e-8`) para não dividir por zero.

**Entrada:** `X (n,d)`  
**Saída:** `X_std (n,d)`, `mean (d,)`, `std (d,)`


In [44]:
def standardize_fit_demo(X):
    """TODO A2: calcule mean, std (axis=0), devolva X_std, mean, std."""
    mean = np.mean(X, axis=0)
    std = np.std(X, axis=0)
    std = np.where(std < 1e-8, 1e-8, std)  # dica
    X_std = (X - mean) / std
    return X_std, mean, std


Xs, m, s = standardize_fit_demo(X_demo)
print("mean", m, "std", s)
print("Xs\n", Xs)
print("nova média (quase zero):", Xs.mean(axis=0))


mean [ 3. 30.] std [ 1.63299316 16.32993162]
Xs
 [[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]
nova média (quase zero): [0. 0.]


Quando A1 e A2 rodarem sem erro, siga. Se `...` der SyntaxError, troque por um valor/código de verdade.


## B. Carregar o Liver e definir o alvo

### O que o arquivo tem

Colunas típicas: `mcv`, `alkphos`, `sgpt`, `sgot`, `gammagt`, `drinks`, `selector`.

- **Features (X):** só os 5 exames. Shape `(345, 5)`.
- **Alvo (y):** **não** use `selector`. Use a regra Turney:
  - `y = 1` se `drinks >= 3`
  - `y = 0` caso contrário  
  Shape `(345,)`.

**Analogia (papéis):**

| Peça | Pergunta | Conteúdo |
| --- | --- | --- |
| `X` | “O que o modelo **vê**?” | 5 exames por pessoa |
| `y` | “O que queremos **adivinhar**?” | 0/1 pela regra dos drinks |

`selector` não é o alvo desta tarefa. Ignore-o ao montar `y`.

### TODO B1

1. `pd.read_csv(DATA_PATH)`
2. Monte `X` e `y` como `np.ndarray` de `float`.


In [45]:
import pandas as pd

DATA_PATH = Path("../../../data/liver.csv")
assert DATA_PATH.exists(), f"Não achei {DATA_PATH.resolve()}. Ative o repo certo."

SEEDS = [42, 7, 13, 21, 99]
TEST_SIZE = 0.30

# TODO B1
df = pd.read_csv(DATA_PATH)
feature_cols = ["mcv", "alkphos", "sgpt", "sgot", "gammagt"]
X = df[feature_cols].to_numpy(dtype=float)
y = (df["drinks"].to_numpy(dtype=float) >= 3).astype(float)

# Quando preencher, descomente:
print(X.shape, y.shape, y.mean())
assert X.shape == (345, 5)
assert y.shape == (345,)


(345, 5) (345,) 0.5101449275362319


## C. Padronizar no treino (e só no treino)

Na entrega você separa treino/teste. A regra de ouro:

1. Calcula `mean` e `std` **só com X_train**
2. Aplica a **mesma** `mean`/`std` em `X_test`

**Analogia (régua):**

| Fase | Pergunta | Ação |
| --- | --- | --- |
| Fit (treino) | “Qual é a régua deste conjunto?” | `mean`, `std` em `X_train` |
| Apply (teste) | “Como meço o teste com a mesma régua?” | `(X - mean) / std` |

### Receita (igual ao A2)

1. `mean = np.mean(X_train, axis=0)`
2. `std = np.std(X_train, axis=0)`
3. `std = np.where(std < 1e-8, 1e-8, std)`
4. Em qualquer matriz: `X_std = (X - mean) / std`

### TODO C1

Se a célula abaixo já estiver preenchida, só rode e siga.


In [46]:
def standardize_fit(X_train: np.ndarray):
    """Entrada: (n_train, d). Saída: mean (d,), std (d,)."""
    mean = np.mean(X_train, axis=0)
    std = np.std(X_train, axis=0)
    std = np.where(std < 1e-8, 1e-8, std)
    return mean, std


def standardize_apply(X: np.ndarray, mean: np.ndarray, std: np.ndarray) -> np.ndarray:
    """Entrada: X (n,d), mean (d,), std (d,). Saída: X_std (n,d)."""
    return (X - mean) / std


## D. Sigmoide — a “porta suave”

Fórmula:

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

- Se `z` é um número ou um array, o NumPy aplica **em cada entrada**.
- `σ(0) = 0.5`
- `z` bem positivo → perto de 1; bem negativo → perto de 0

**Conexão `z` e `a`:** `a = σ(z)`. São o “antes” e o “depois” da porta suave.

Derivada útil quando você já tem `a = σ(z)`:

$$
\sigma'(z) = a(1-a)
$$

**Analogia (fases — grave isto):**

| Fase | Pergunta | Ferramenta |
| --- | --- | --- |
| Forward (previsão) | “Qual a saída deste neurônio?” | `σ(z)` → `a` |
| Backward (aprendizado) | “Se eu mexer um pouco em `z`, a saída muda quanto?” | `σ'(z) = a(1-a)` |

A sigmoide **produz** a ativação. A derivada **guia** o ajuste dos pesos. Não é uma no lugar da outra — cada uma entra num momento diferente.

Exemplo: `z = 0` → `a = 0.5` → derivada `0.5*(1-0.5) = 0.25`.

### TODO D1

Implemente `sigmoid` e `sigmoid_deriv_from_a`.  
**Dica numérica:** `np.clip(z, -500, 500)` antes do `exp` evita overflow (corta extremos para o `exp` não estourar).


In [47]:
def sigmoid(z: np.ndarray) -> np.ndarray:
    z = np.clip(z, -500, 500)
    
    return 1.0 / (1.0 + np.exp(-z))


def sigmoid_deriv_from_a(a: np.ndarray) -> np.ndarray:    
    return a * (1 - a)


# Testes — descomente quando D1 estiver pronto:
assert abs(float(sigmoid(0.0)) - 0.5) < 1e-6
assert float(sigmoid(20.0)) > 0.99
assert abs(float(sigmoid_deriv_from_a(0.5)) - 0.25) < 1e-6
print("sigmoid OK")


sigmoid OK


## E. Forward — uma camada oculta

Arquitetura:

- Entrada: `d = 5` (exames)
- Oculta: `H` neurônios (comece com `H = 10`)
- Saída: 1 número entre 0 e 1 (probabilidade da classe 1)

**Analogia (montagem):**

| Camada | Pergunta | Conta |
| --- | --- | --- |
| Oculta | “Que misturas dos 5 exames são úteis?” | `a1 = σ(X @ W1 + b1)` |
| Saída | “Com essas misturas, qual a chance da classe 1?” | `y_hat = σ(a1 @ W2 + b2)` |

### Receita (você escreve)

1. `z1 = X @ W1 + b1` → shape `(n, H)`
2. `a1 = sigmoid(z1)` → `(n, H)`
3. `z2 = a1 @ W2 + b2` → `(n, 1)`
4. `y_hat = sigmoid(z2)` → `(n, 1)`
5. `cache = {"X": X, "a1": a1, "y_hat": y_hat}` e `return y_hat, cache`

### Shapes

| Peça | Shape |
| --- | --- |
| `W1` | `(d, H)` — use `rng.normal(scale=0.1, size=(d, H))` |
| `b1` | `(H,)` — comece com zeros |
| `W2` | `(H, 1)` |
| `b2` | `(1,)` |
| pesos totais | `H*(d+1) + (H+1)` → se `d=5`, `H=10` → **71** |

### TODO E1

Preencha os `...`. Descomente o teste no final quando achar que está pronto.


In [48]:
def init_weights(d: int, H: int, rng: np.random.Generator):
    """Devolve dict com W1, b1, W2, b2."""
    # TODO E1 — shapes: W1 (d,H), b1 (H,), W2 (H,1), b2 (1,)
    W1 = rng.normal(scale=0.1, size=(d, H)) # rng.normal(scale=0.1, size=(d, H))
    b1 = np.zeros(H)
    W2 = rng.normal(scale=0.1, size=(H, 1))
    b2 = np.zeros(1)
    return {"W1": W1, "b1": b1, "W2": W2, "b2": b2}


def forward(X: np.ndarray, params: dict):
    """Retorna (y_hat, cache)."""
    # TODO E1 — siga a receita da célula markdown (z1 → a1 → z2 → y_hat)
    z1 = X @ params['W1'] + params["b1"]
    a1 = sigmoid(z1)
    z2 = a1 @ params['W2'] + params['b2']
    y_hat = sigmoid(z2)
    cache = {"X": X, "a1": a1, "y_hat": y_hat}
    return y_hat, cache


def count_parameters(d: int, H: int) -> int:
    # TODO E1 — H*(d+1) + (H+1)
    return H*(d+1) + (H+1)


# Quando preencher, descomente:
_rng = np.random.default_rng(0)
_p = init_weights(5, 10, _rng)
_yh, _ = forward(np.zeros((4, 5)), _p)
print(_yh.shape, count_parameters(5, 10))
assert count_parameters(5, 10) == 71
assert _yh.shape == (4, 1)
print("E1 OK")


(4, 1) 71
E1 OK


## F. Loss (erro “contínuo”) e erro de classificação

**Binary cross-entropy** (treino):

$$
L = -\frac{1}{n}\sum_i \big[ y_i \log \hat{y}_i + (1-y_i)\log(1-\hat{y}_i) \big]
$$

**Erro de classificação** (placar da entrega):

$$
\mathrm{erro} = 1 - \mathrm{acurácia}
\quad\text{com}\quad
\hat{y}^{\mathrm{cls}}=1 \text{ se } \hat{y}\ge 0.5.
$$

**Analogia (dois relógios):**

| Métrica | Pergunta | Quando usar |
| --- | --- | --- |
| BCE | “Quão longe a **probabilidade** está do alvo?” | treino |
| Erro 0/1 | “Quantas vezes errei o **sim/não**?” | tabela S |

### Receita BCE (você escreve)

1. `y = y_true.reshape(-1)` e `p = y_hat.reshape(-1)`
2. `p = np.clip(p, eps, 1 - eps)`
3. `return float( -np.mean( y*np.log(p) + (1-y)*np.log(1-p) ) )`

### Receita erro de classificação

1. Achate `y` e `p`
2. `pred = (p >= threshold).astype(float)`
3. `return float(np.mean(pred != y))`

### TODO F1

Preencha. Descomente os asserts no final.


In [49]:
def binary_cross_entropy(y_true: np.ndarray, y_hat: np.ndarray, eps: float = 1e-9) -> float:
    # TODO F1 — receita da célula markdown
    y = y_true.reshape(-1)
    p = y_hat.reshape(-1)
    p = np.clip(p, eps, 1- eps)
    return float(-np.mean(y * np.log(p) + (1-y) * np.log(1-p)))


def classification_error(y_true: np.ndarray, y_hat: np.ndarray, threshold: float = 0.5) -> float:
    # TODO F1
    y = y_true.reshape(-1)
    p = y_hat.reshape(-1)
    pred = (p >= threshold).astype(float)
    return float(np.mean(pred != y))


# Quando preencher, descomente:
assert abs(binary_cross_entropy(np.array([1.0]), np.array([0.9])) - (-np.log(0.9))) < 1e-6
assert classification_error(np.array([1.0, 0.0]), np.array([0.9, 0.1])) == 0.0
print("F1 OK")


F1 OK


## G. Backward + um passo de SGD

### Ideia (sem terror)

1. Forward → `y_hat`
2. Loss → quão errado
3. Backward → “para onde empurrar cada peso?”
4. SGD → `peso ← peso - lr * gradiente`

**Analogia:**

| Fase | Pergunta | Ferramenta |
| --- | --- | --- |
| Forward | “Qual a previsão?” | `forward` |
| Backward | “Para onde empurrar?” | `backward` |
| SGD | “Um passo nessa direção.” | `params[k] - lr * grads[k]` |

### Receita backward (BCE + sigmoide na saída)

Use estas contas (média sobre `n` exemplos). Você traduz para NumPy:

1. `y = y_true.reshape(-1, 1)` e `y_hat = y_hat.reshape(-1, 1)`
2. `delta2 = (y_hat - y) / n` → shape `(n, 1)`
3. `dW2 = a1.T @ delta2` → `(H, 1)`
4. `db2 = np.sum(delta2, axis=0)` → `(1,)`
5. `delta1 = (delta2 @ W2.T) * a1 * (1 - a1)` → `(n, H)`  
   (o `*` é a derivada da sigmoide: `a(1-a)`)
6. `dW1 = X.T @ delta1` → `(d, H)`
7. `db1 = np.sum(delta1, axis=0)` → `(H,)`
8. `return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}`

`X` e `a1` vêm do `cache`. `n = X.shape[0]`.

### Receita `sgd_step`

Para cada chave `k` em `params`: `params[k] = params[k] - lr * grads[k]`

### Receita `train_mlp`

1. `params = init_weights(...)`
2. Loop `epochs`: forward → loss → backward → sgd_step; guarde a loss
3. No fim: erro de classificação no **teste**

### TODO G1 / G2

Preencha. O smoke test no final só roda quando você descomentar.


In [50]:
def backward(y_true: np.ndarray, y_hat: np.ndarray, cache: dict, params: dict) -> dict:
    """grads com as mesmas chaves de params."""
    # TODO G1 — receita da célula markdown
    X = cache["X"]
    a1 = cache["a1"]
    n = X.shape[0]
    W2 = params['W2']

    y = y_true.reshape(-1, 1)
    y_hat = y_hat.reshape(-1, 1)

    delta2 = (y_hat - y) / n
    dW2 = a1.T @ delta2
    db2 = np.sum(delta2, axis=0)

    delta1 = (delta2 @ W2.T) * a1 * (1 - a1)
    dW1 = X.T @ delta1
    db1 = np.sum(delta1, axis=0)

    return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}


def sgd_step(params: dict, grads: dict, lr: float) -> None:
    # TODO G1 — para cada chave: params[k] = params[k] - lr * grads[k]
    for k in params:
        params[k] = params[k] - lr * grads[k]


def train_mlp(X_train, y_train, X_test, y_test, H=10, lr=0.1, epochs=200, seed=0):
    """TODO G2: init → loop (forward, loss, backward, sgd) → erro no teste."""
    rng = np.random.default_rng(seed)
    d = X_train.shape[1]
    params = init_weights(d, H, rng)
    history = []

    for _ in range(epochs):
        # TODO G2 — quatro linhas: forward, loss, backward, sgd_step
        y_hat, cache = forward(X_train, params)
        loss = binary_cross_entropy(y_train, y_hat)
        grads = backward(y_train, y_hat, cache, params)
        sgd_step(params, grads, lr)
        history.append(loss)  # a loss

    # TODO G2 — forward no teste + classification_error
    y_test_hat, _ = forward(X_test, params)
    test_err = classification_error(y_test, y_test_hat)
    return params, history, test_err


# Quando G1/G2 estiver pronto, descomente (a loss deve DESCER):
_rng = np.random.default_rng(1)
_Xtr = _rng.normal(size=(40, 5)); _ytr = (_rng.random(40) > 0.5).astype(float)
_Xte = _rng.normal(size=(10, 5)); _yte = (_rng.random(10) > 0.5).astype(float)
_params, _hist, _err = train_mlp(_Xtr, _ytr, _Xte, _yte, H=8, lr=0.2, epochs=50, seed=1)
print("loss", _hist[0], "→", _hist[-1], "erro teste", _err)
assert _hist[-1] < _hist[0]
print("G1/G2 OK")


loss 0.7045456018129703 → 0.6846624775834668 erro teste 0.6
G1/G2 OK


## H. Protocolo da entrega (5 repetições)

Para cada `seed` em `SEEDS = [42, 7, 13, 21, 99]`:

1. Split **70% / 30%** (estratificado por `y`, `random_state=seed`)
2. `mean, std = standardize_fit(X_train)` → `standardize_apply` nos dois
3. `train_mlp(...)`
4. Guarde o erro de classificação no **teste**
5. Salve CSVs em `partitions/rep{k}/`

**Analogia:** treino = ensaio; teste = prova; 5 seeds = 5 sorteios.

Pode usar `train_test_split` **só para o split** (não para a MLP).

### TODO H1

Preencha o loop. A estrutura do `for` já está esboçada.


In [51]:
from sklearn.model_selection import train_test_split

PART_DIR = Path("partitions")
PART_DIR.mkdir(exist_ok=True)


def run_protocol(X, y, H=10, lr=0.1, epochs=200):
    """TODO H1: devolve lista com 5 erros de teste; grava partições."""
    errors = []
    for k, seed in enumerate(SEEDS, start=1):
        # 1) split
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=TEST_SIZE, random_state=seed, stratify=y
        )
        # 2) padronizar (fit só no treino)
        mean, std = standardize_fit(X_tr)
        X_tr_s = standardize_apply(X_tr, mean, std)
        X_te_s = standardize_apply(X_te, mean, std)
        # 3) treinar
        params, history, test_err = train_mlp(
            X_tr_s, y_tr, X_te_s, y_te, H=H, lr=lr, epochs=epochs, seed=seed
        )
        errors.append(test_err)
        # 4) salvar CSVs
        rep = PART_DIR / f"rep{k}"
        rep.mkdir(exist_ok=True)
        np.savetxt(rep / "X_train.csv", X_tr, delimiter=",")
        np.savetxt(rep / "X_test.csv", X_te, delimiter=",")
        np.savetxt(rep / "y_train.csv", y_tr, delimiter=",")
        np.savetxt(rep / "y_test.csv", y_te, delimiter=",")
        # opcional: meta.json com seed, mean, std
        print(f"rep{k} seed={seed} erro_teste={test_err:.4f}")
    return errors


# Quando H1 estiver pronto:
errors = run_protocol(X, y)
print("erro médio:", float(np.mean(errors)))



rep1 seed=42 erro_teste=0.4808
rep2 seed=7 erro_teste=0.4231
rep3 seed=13 erro_teste=0.5096
rep4 seed=21 erro_teste=0.4038
rep5 seed=99 erro_teste=0.4135
erro médio: 0.44615384615384607


## S. Comparação (preencha depois de rodar 1–3)

Mesmo protocolo: 5 seeds `[42,7,13,21,99]`, 70/30 estratificado, alvo Turney, features padronizadas no treino.

| Modelo | Erro médio (teste) |
| --- | --- |
| MLP NumPy (este notebook) | ~0,45 |
| MLP PyTorch | |
| MLP scikit-learn | |
| Árvore (Tarefa 2) | 0,4327 |
| Naïve Bayes (Tarefa 2) | 0,4346 |
| k-NN (Tarefa 2) | 0,4423 |
| Ensemble (Tarefa 2) | 0,4365 |

**Discussão (você escreve):** quem generalizou melhor? Overfitting? Escala / `H` / `lr` / `max_iter`?

*Conteúdo, experimentos e conclusões são meus; a formatação do texto teve assistência de IA.*


## Bloco 2 — PyTorch (mesmo problema, bem menos código)

**Ideia:** você não escreve `backward` na mão. Monta a rede, escolhe loss + otimizador, e o autograd deriva.

Instalação (uma vez, venv da raiz):

```bash
uv sync
source .venv/bin/activate
```

Torch está nas dependências padrão do `pyproject.toml` na raiz.

### Anatomia mínima

1. `nn.Linear(d, H)` + `nn.ReLU` (ou `Sigmoid`) + `nn.Linear(H, 1)`
2. `nn.BCEWithLogitsLoss` (mais estável) **ou** `BCELoss` com `Sigmoid` na saída
3. `optim.SGD` ou `Adam`
4. Loop: `zero_grad` → `forward` → `loss` → `backward` → `step`

Shapes: `X` → `torch.tensor` float32 `(n, d)`; `y` → `(n, 1)`.


In [52]:
# Smoke: torch instalado?
try:
    import torch
    import torch.nn as nn
    print("torch", torch.__version__)
except ImportError:
    print("Instale torch no .venv da raiz (comando na célula markdown acima).")
    torch = None
    nn = None


torch 2.14.0


### TODO T1 — definir a rede e treinar em **uma** partição

Não precisa ser idêntico ao NumPy (`H=10`, sigmoide nas duas camadas). Pode usar ReLU na oculta + logits na saída — **documente** no relatório o que usou.

Complete a classe e o loop. Meta: loss desce num split só.


In [53]:
if torch is not None:

    class MLPTorch(nn.Module):
        def __init__(self, d: int, H: int = 10):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(d, H),
                nn.ReLU(),
                nn.Linear(H, 1),  # logit
            )

        def forward(self, x):
            return self.net(x)

    def train_mlp_torch(X_train, y_train, X_test, y_test, H=10, lr=1e-2, epochs=200, seed=0):
        torch.manual_seed(seed)
        d = X_train.shape[1]
        model = MLPTorch(d, H)
        opt = torch.optim.Adam(model.parameters(), lr=lr)
        loss_fn = nn.BCEWithLogitsLoss()
        Xt = torch.tensor(X_train, dtype=torch.float32)
        yt = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32)
        
        for _ in range(epochs):
            model.train()
            opt.zero_grad()
            logits = model(Xt)
            loss = loss_fn(logits, yt)
            loss.backward()
            opt.step()
        
        X_te_t = torch.tensor(X_test, dtype=torch.float32)
        y_te = y_test.reshape(-1)
        model.eval()
        with torch.no_grad():
            probs = torch.sigmoid(model(X_te_t)).numpy().reshape(-1)
            pred = (probs >= 0.5).astype(float)
        test_err = float(np.mean(pred != y_te))
        return test_err
        

    def run_protocol_torch(X, y, H=10, lr=1e-2, epochs=200):
        """Mesmas SEEDS / split / standardize do NumPy; lista de 5 erros."""
        errors = []
        for k, seed in enumerate(SEEDS, start=1):
            X_tr, X_te, y_tr, y_te = train_test_split(
                X, y, test_size=TEST_SIZE, random_state=seed, stratify=y
            )
            mean, std = standardize_fit(X_tr)
            X_tr_s = standardize_apply(X_tr, mean, std)
            X_te_s = standardize_apply(X_te, mean, std)
            test_err = train_mlp_torch(
                X_tr_s, y_tr, X_te_s, y_te, H=H, lr=lr, epochs=epochs, seed=seed
            )
            errors.append(test_err)
            print(f"[torch] rep{k} seed={seed} erro_teste={test_err:.4f}")
        return errors
    errors_torch = run_protocol_torch(X, y)
    print("erro médio torch:", float(np.mean(errors_torch)))
else:
    print("Pulando stubs torch até instalar.")


[torch] rep1 seed=42 erro_teste=0.4135
[torch] rep2 seed=7 erro_teste=0.4519
[torch] rep3 seed=13 erro_teste=0.3462
[torch] rep4 seed=21 erro_teste=0.4615
[torch] rep5 seed=99 erro_teste=0.4808
erro médio torch: 0.43076923076923085


## Bloco 3 — scikit-learn (`MLPClassifier`)

Aqui a “ferramenta com MLP” do enunciado fica explícita: quase só API.

```python
from sklearn.neural_network import MLPClassifier
clf = MLPClassifier(hidden_layer_sizes=(10,), activation="logistic",  # ~sigmoide
                    max_iter=500, random_state=seed)
clf.fit(X_tr_s, y_tr)
pred = clf.predict(X_te_s)
erro = np.mean(pred != y_te)
```

### TODO S1 — `run_protocol_sklearn`

Mesmas seeds, split, padronização. Documente no relatório: versão do sklearn + hiperparâmetros.


In [54]:
from sklearn.neural_network import MLPClassifier


def run_protocol_sklearn(X, y, hidden=(10,), max_iter=500):
    """5 erros de teste; mesmo protocolo do NumPy."""
    errors = []
    for k, seed in enumerate(SEEDS, start=1):
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=TEST_SIZE, random_state=seed, stratify=y
        )
        mean, std = standardize_fit(X_tr)
        X_tr_s = standardize_apply(X_tr, mean, std)
        X_te_s = standardize_apply(X_te, mean, std)

        clf = MLPClassifier(
            hidden_layer_sizes=hidden,
            activation="logistic",
            solver="adam",
            max_iter=max_iter,
            random_state=seed,
        )
        clf.fit(X_tr_s, y_tr)
        pred = clf.predict(X_te_s)
        test_err = float(np.mean(pred != y_te))
        errors.append(test_err)
        print(f"[sklearn] rep{k} seed={seed} erro_teste={test_err:.4f}")
    return errors


errors_sk = run_protocol_sklearn(X, y)
print("erro médio sklearn:", float(np.mean(errors_sk)))

[sklearn] rep1 seed=42 erro_teste=0.3365
[sklearn] rep2 seed=7 erro_teste=0.4135
[sklearn] rep3 seed=13 erro_teste=0.3365
[sklearn] rep4 seed=21 erro_teste=0.4712
[sklearn] rep5 seed=99 erro_teste=0.4712
erro médio sklearn: 0.4057692307692308


## Próximos micro-passos (hoje → amanhã)

1. Completar CSVs em `partitions/rep{{k}}/` (`X_test`, `y_train`, `y_test`).
2. Preencher **T1/T2** (Torch) e **S1** (sklearn) — mesmo protocolo.
3. Preencher tabela **S** e 1 parágrafo de discussão.
4. Render PDF (Quarto) + ZIP em `courses/ft108a-machine-learning/entregas/`.

Ordem boa se estiver cansado: **sklearn primeiro** (mais curto) → Torch → PDF.
